In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 1) import libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
from sklearn.feature_selection import VarianceThreshold, mutual_info_regression
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
import lightgbm as lgb
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt


In [3]:
# 2) load CSV
path = "/content/drive/MyDrive/data-organisation-fme-2026/librosa_features_full_fme_dataset_MARCH_2026_NC.csv"
df = pd.read_csv(path)

In [5]:
# optional step - filter out to musicians only

# load Gold-MSI demographics


demo_path = "/content/drive/MyDrive/data-organisation-fme-2026/fme-demographics-music-sophistication.csv"
demo = pd.read_csv(demo_path)

print("Demographics shape:", demo.shape)

# responses considered musicians
musician_answers = [
    "sort-of-agree",
    "mostly-agree",
    "strongly-agree"
]

# identify musician participants
musicians = demo[
    demo["musicMusician"].isin(musician_answers)
]["participant-ID"].unique()

print("Musician participants:", len(musicians))
print("Example musician IDs:", musicians[:10])
# 59/98 are musicians, 9 not sure, 30 non musicians

df = df[df["participant"].isin(musicians)].copy()

print("\nAfter filtering musicians:")
print("Rows:", len(df))
print("Participants:", df["participant"].nunique())
print("Songs:", df["file_path"].nunique())

print("MUSICIAN FILTER SUMMARY:")

total_participants = demo["participant-ID"].nunique()
musician_count = len(musicians)

print("Total participants:", total_participants)
print("Musicians:", musician_count)
print("Non-musicians:", total_participants - musician_count)

print("Annotations retained:", len(df))

Demographics shape: (115, 27)
Musician participants: 59
Example musician IDs: ['_0sca4vavd' '_14q3qbw23' '_1cpqpmytm' '_57v8oow2l' '_5wyte9tjg'
 '_68v3ai68u' '_6pyclh0uk' '_7twr1orir' '_87gyh97l5' '_92wls1kek']

After filtering musicians:
Rows: 1045
Participants: 57
Songs: 310
MUSICIAN FILTER SUMMARY:
Total participants: 100
Musicians: 59
Non-musicians: 41
Annotations retained: 1045


In [6]:

# 3) cluster timestamps (3-sec windows)
df['cluster'] = (df['timestamp_imputed'] // 3).astype(int)
df['valence'] = df['valence'].round(3)
df['arousal'] = df['arousal'].round(3)

df_clustered = df.groupby(['file_path','cluster']).agg({
    'valence':'mean',
    'arousal':'mean'
}).reset_index()


# 4) feature processing

meta_cols = ['file_path','cluster']
target_cols = ['valence','arousal']
feature_cols = [c for c in df.columns if c not in meta_cols + target_cols]

def fix_value(v):
    if isinstance(v,(int,float,np.number)): return v
    if isinstance(v,str):
        v = v.strip()
        if v.startswith("[") and v.endswith("]"):
            try: return float(v[1:-1])
            except: return np.nan
        try: return float(v)
        except: return np.nan
    return np.nan

for col in feature_cols:
    df[col] = df[col].apply(fix_value)

# merge features with clustered targets
df_clustered = df_clustered.merge(df[meta_cols + feature_cols], on='file_path', how='left')
df_clustered = df_clustered.dropna(subset=['valence','arousal'])

X = df_clustered[feature_cols].values
y_val = df_clustered['valence'].values
y_ar = df_clustered['arousal'].values

# impute + scale
X = KNNImputer(n_neighbors=3).fit_transform(X)
X = StandardScaler().fit_transform(X)

# feature selection: variance + mutual info
X = VarianceThreshold(0.0).fit_transform(X)
mi_avg = (mutual_info_regression(X, y_val) + mutual_info_regression(X, y_ar)) / 2
top_idx = np.argsort(mi_avg)[-40:]
X = X[:, top_idx]


# 5) train/test split
X_train, X_test, yv_train, yv_test, ya_train, ya_test = train_test_split(
    X, y_val, y_ar, test_size=0.2, random_state=42
)


# 6) models
def build_dnn(input_dim):
    model = Sequential([
        Dense(128, input_dim=input_dim, activation='relu'),
        BatchNormalization(),
        Dropout(0.3),
        Dense(64, activation='relu'),
        BatchNormalization(),
        Dropout(0.2),
        Dense(1, activation='linear')
    ])
    model.compile(optimizer=Adam(1e-3), loss='mse')
    return model

rf = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
lgb_val = lgb.LGBMRegressor(n_estimators=2000, learning_rate=0.01, num_leaves=80, random_state=42)
lgb_ar = lgb.LGBMRegressor(n_estimators=2000, learning_rate=0.01, num_leaves=80, random_state=42)

# train RF
rf.fit(X_train, yv_train)
pred_val_rf = rf.predict(X_test)
rf.fit(X_train, ya_train)
pred_ar_rf = rf.predict(X_test)

# train LGB
lgb_val.fit(X_train, yv_train)
pred_val_lgb = lgb_val.predict(X_test)
lgb_ar.fit(X_train, ya_train)
pred_ar_lgb = lgb_ar.predict(X_test)

# train DNN
dnn_val = build_dnn(X_train.shape[1])
es = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
dnn_val.fit(X_train, yv_train, validation_split=0.2, epochs=200, batch_size=32, verbose=0, callbacks=[es])
pred_val_dnn = dnn_val.predict(X_test).flatten()

dnn_ar = build_dnn(X_train.shape[1])
dnn_ar.fit(X_train, ya_train, validation_split=0.2, epochs=200, batch_size=32, verbose=0, callbacks=[es])
pred_ar_dnn = dnn_ar.predict(X_test).flatten()

# hybrid ensemble
pred_val_hybrid = 0.5*pred_val_lgb + 0.2*pred_val_rf + 0.3*pred_val_dnn
pred_ar_hybrid = 0.5*pred_ar_lgb + 0.2*pred_ar_rf + 0.3*pred_ar_dnn

# 7) evaluation:

def evaluate(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    tol_acc = np.mean(np.abs(y_true - y_pred) < 0.2)
    print(f"{name}: MAE={mae:.3f}, RMSE={rmse:.3f}, R²={r2:.3f}, Within ±0.2={tol_acc*100:.1f}%")

for name, y, pred in [
    ("RF Valence", yv_test, pred_val_rf),
    ("RF Arousal", ya_test, pred_ar_rf),
    ("LGB Valence", yv_test, pred_val_lgb),
    ("LGB Arousal", ya_test, pred_ar_lgb),
    ("DNN Valence", yv_test, pred_val_dnn),
    ("DNN Arousal", ya_test, pred_ar_dnn),
    ("Hybrid Valence", yv_test, pred_val_hybrid),
    ("Hybrid Arousal", ya_test, pred_ar_hybrid)
]:
    evaluate(y, pred, name)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003086 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 9348
[LightGBM] [Info] Number of data points in the train set: 2810, number of used features: 40
[LightGBM] [Info] Start training from score 0.078000


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001220 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 9348
[LightGBM] [Info] Number of data points in the train set: 2810, number of used features: 40
[LightGBM] [Info] Start training from score 0.167642


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
RF Valence: MAE=0.239, RMSE=0.314, R²=0.369, Within ±0.2=53.1%
RF Arousal: MAE=0.226, RMSE=0.303, R²=0.419, Within ±0.2=57.0%
LGB Valence: MAE=0.245, RMSE=0.328, R²=0.311, Within ±0.2=53.6%
LGB Arousal: MAE=0.235, RMSE=0.318, R²=0.357, Within ±0.2=54.9%
DNN Valence: MAE=0.248, RMSE=0.320, R²=0.344, Within ±0.2=50.2%
DNN Arousal: MAE=0.238, RMSE=0.298, R²=0.436, Within ±0.2=50.5%
Hybrid Valence: MAE=0.236, RMSE=0.313, R²=0.372, Within ±0.2=54.8%
Hybrid Arousal: MAE=0.224, RMSE=0.296, R²=0.442, Within ±0.2=57.3%


In [9]:
# evaluate models on full dataset:::::::::::
# 2) load CSV
path = "/content/drive/MyDrive/data-organisation-fme-2026/librosa_features_full_fme_dataset_MARCH_2026_NC.csv"
df = pd.read_csv(path)

# NO musician filtering — full dataset:

print("Full dataset:")
print("Rows:", len(df))
print("Participants:", df["participant"].nunique())
print("Songs:", df["file_path"].nunique())

# 3) cluster timestamps (3-sec windows)
df['cluster'] = (df['timestamp_imputed'] // 3).astype(int)
df['valence'] = df['valence'].round(3)
df['arousal'] = df['arousal'].round(3)

df_clustered = df.groupby(['file_path','cluster']).agg({
    'valence':'mean',
    'arousal':'mean'
}).reset_index()

# 4) feature processing
meta_cols = ['file_path','cluster']
target_cols = ['valence','arousal']
feature_cols = [c for c in df.columns if c not in meta_cols + target_cols]

def fix_value(v):
    if isinstance(v,(int,float,np.number)): return v
    if isinstance(v,str):
        v = v.strip()
        if v.startswith("[") and v.endswith("]"):
            try: return float(v[1:-1])
            except: return np.nan
        try: return float(v)
        except: return np.nan
    return np.nan

for col in feature_cols:
    df[col] = df[col].apply(fix_value)

# merge features with clustered targets
df_clustered = df_clustered.merge(df[meta_cols + feature_cols], on='file_path', how='left')
df_clustered = df_clustered.dropna(subset=['valence','arousal'])

X = df_clustered[feature_cols].values
y_val = df_clustered['valence'].values
y_ar = df_clustered['arousal'].values

# impute + scale
X = KNNImputer(n_neighbors=3).fit_transform(X)
X = StandardScaler().fit_transform(X)

# feature selection: variance + mutual info
X = VarianceThreshold(0.0).fit_transform(X)
mi_avg = (mutual_info_regression(X, y_val) + mutual_info_regression(X, y_ar)) / 2
top_idx = np.argsort(mi_avg)[-40:]
X = X[:, top_idx]

# 5) train/test split
X_train, X_test, yv_train, yv_test, ya_train, ya_test = train_test_split(
    X, y_val, y_ar, test_size=0.2, random_state=42
)

# 6) models
def build_dnn(input_dim):
    model = Sequential([
        Dense(128, input_dim=input_dim, activation='relu'),
        BatchNormalization(),
        Dropout(0.3),
        Dense(64, activation='relu'),
        BatchNormalization(),
        Dropout(0.2),
        Dense(1, activation='linear')
    ])
    model.compile(optimizer=Adam(1e-3), loss='mse')
    return model

rf = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
lgb_val = lgb.LGBMRegressor(n_estimators=2000, learning_rate=0.01, num_leaves=80, random_state=42)
lgb_ar = lgb.LGBMRegressor(n_estimators=2000, learning_rate=0.01, num_leaves=80, random_state=42)

# train RF
rf.fit(X_train, yv_train)
pred_val_rf = rf.predict(X_test)
rf.fit(X_train, ya_train)
pred_ar_rf = rf.predict(X_test)

# train LGB
lgb_val.fit(X_train, yv_train)
pred_val_lgb = lgb_val.predict(X_test)
lgb_ar.fit(X_train, ya_train)
pred_ar_lgb = lgb_ar.predict(X_test)

# train DNN
dnn_val = build_dnn(X_train.shape[1])
es = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
dnn_val.fit(X_train, yv_train, validation_split=0.2, epochs=200, batch_size=32, verbose=0, callbacks=[es])
pred_val_dnn = dnn_val.predict(X_test).flatten()

dnn_ar = build_dnn(X_train.shape[1])
dnn_ar.fit(X_train, ya_train, validation_split=0.2, epochs=200, batch_size=32, verbose=0, callbacks=[es])
pred_ar_dnn = dnn_ar.predict(X_test).flatten()

# hybrid ensemble
pred_val_hybrid = 0.5*pred_val_lgb + 0.2*pred_val_rf + 0.3*pred_val_dnn
pred_ar_hybrid = 0.5*pred_ar_lgb + 0.2*pred_ar_rf + 0.3*pred_ar_dnn

# 7) evaluation
def evaluate(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    tol_acc = np.mean(np.abs(y_true - y_pred) < 0.2)
    print(f"{name}: MAE={mae:.3f}, RMSE={rmse:.3f}, R²={r2:.3f}, Within ±0.2={tol_acc*100:.1f}%")

for name, y, pred in [
    ("RF Valence", yv_test, pred_val_rf),
    ("RF Arousal", ya_test, pred_ar_rf),
    ("LGB Valence", yv_test, pred_val_lgb),
    ("LGB Arousal", ya_test, pred_ar_lgb),
    ("DNN Valence", yv_test, pred_val_dnn),
    ("DNN Arousal", ya_test, pred_ar_dnn),
    ("Hybrid Valence", yv_test, pred_val_hybrid),
    ("Hybrid Arousal", ya_test, pred_ar_hybrid)
]:
    evaluate(y, pred, name)

Full dataset:
Rows: 1784
Participants: 98
Songs: 422
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002459 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 9364
[LightGBM] [Info] Number of data points in the train set: 5256, number of used features: 40
[LightGBM] [Info] Start training from score 0.061667


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002629 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 9364
[LightGBM] [Info] Number of data points in the train set: 5256, number of used features: 40
[LightGBM] [Info] Start training from score 0.186004


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
RF Valence: MAE=0.248, RMSE=0.321, R²=0.359, Within ±0.2=50.3%
RF Arousal: MAE=0.239, RMSE=0.321, R²=0.302, Within ±0.2=53.8%
LGB Valence: MAE=0.252, RMSE=0.328, R²=0.327, Within ±0.2=50.3%
LGB Arousal: MAE=0.244, RMSE=0.332, R²=0.256, Within ±0.2=53.6%
DNN Valence: MAE=0.242, RMSE=0.309, R²=0.406, Within ±0.2=50.3%
DNN Arousal: MAE=0.351, RMSE=0.446, R²=-0.347, Within ±0.2=35.5%
Hybrid Valence: MAE=0.242, RMSE=0.313, R²=0.390, Within ±0.2=51.9%
Hybrid Arousal: MAE=0.249, RMSE=0.325, R²=0.285, Within ±0.2=49.5%
